# Problem Scoping

Defines the two Milestone 2 problems, their stakeholder and success criteria, and the leakage rules that every later notebook follows. The full write-up is in Sections 1 and 2.1 of the Milestone 2 report.

**Business question.** How can information available at order placement be used to refine the platform's existing delivery estimate and indicate whether an order is at risk of using or exceeding its promised delivery window?

**Stakeholder.** The platform's customer success team, which answers customers' delivery questions and decides which orders need follow-up. Customers benefit from a more realistic arrival estimate: in the Milestone 1 analysis, late orders averaged 2.57 review stars against 4.29 for on-time orders.

**Prediction point.** Order placement, once checkout information and the platform's estimated delivery date are recorded. The platform's estimate remains the customer commitment; both models provide complementary decision support.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.config import RANDOM_STATE, set_seed

set_seed()

## 1. Problem A — Promised delivery band (classification)
- **Notebooks:** `02_preprocessing_problemA` → `03_feature_engineering_problemA` → `04_classification_promise_band_modeling`.
- **Target:** three bands from the ratio $r$ = realised duration / promised duration: `under_half` ($r < 0.5$), `half_to_full` ($0.5 \le r \le 1$) and `late` ($r > 1$). Class shares are 57.7%, 34.2% and 8.1% of 96,470 delivered orders. The promised date is stored at midnight, so a delivery later on the promised calendar day counts as `late`.
- **Why bands:** they show how much of the promised window an order uses, which tells the customer success team more than a binary late flag. Problem A was first scoped as binary late delivery (`is_late`) and then reframed as these bands.
- **Success criteria:** models are selected by temporal cross-validated Macro-F1, with late-class precision, recall, F1 and average precision reported for the minority band. Accuracy is not used because the classes are imbalanced.

## 2. Problem B — Delivery time (regression)
- **Notebooks:** `02_preprocessing` → `03_feature_engineering` → `05_regression_delivery_time`.
- **Target:** `delivery_days` = days from the purchase timestamp to delivery to the customer, in fractional days. Models learn the log ratio of delivery days to `promised_delivery_days`, which is less exposed to the downward drift in delivery time, and predictions are converted back to days. `freight_value` is an input feature, known at purchase.
- **Success criteria:** MAE and RMSE in days below the promise baseline, `promised_delivery_days` × the median training ratio of delivery days to promised days.

## 3. Prediction-time cut-off and leakage rules
Both problems predict at the moment of purchase. Excluded from features:

- Known only after purchase: `order_approved_at`, `shipping_limit_date`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_status`.
- Outcomes and what follows them: `delay_days`, `delivery_status`, `review_score`. Neither problem's target, nor anything derived from it, is used as a feature.
- History features use only earlier orders. Outcome-based histories (seller and route for both problems; also customer, category and marketplace for Problem A) include an earlier order only if its delivery was observed before the current purchase. Customer histories use `customer_unique_id`, not the order-specific `customer_id`.
- Splits are by purchase time:
  - Problem B: train to 2018-06-30, validation 2018-07, test 2018-08 (83,947 / 6,156 / 6,351 orders).
  - Problem A: cutoffs at the 70th and 85th percentiles of purchase time, with a label-maturity rule: an order enters training or validation only if it was delivered before the next cutoff (64,429 train, 5,029 excluded, 12,541 validation, 14,471 test).
- Preprocessing that learns from data (imputation, scaling, encoding) is fitted on training data only, inside the model pipeline.

`notebooks/02_preprocessing.ipynb` asserts that none of the excluded columns reach the exported tables.